# 17. Real Paper Reproduction Case Study — Original U-Net for Biomedical Segmentation

**Paper:** Ronneberger, Fischer & Brox, *U-Net: Convolutional Networks for Biomedical Image Segmentation* (MICCAI, 2015)  
**Original release:** Freiburg U-Net / modified Caffe + MATLAB tooling  
**Goal:** understand what the original paper actually implemented, reproduce the core segmentation logic in a modern transparent example, and decide how to adapt U-Net to OCT, MUSE, fluorescence, and LSFM.

> Important: a modern `padding="same"` PyTorch U-Net is **not exactly the 2015 network**. This notebook separates **historical reproduction** from **modern research implementation**.

## Mind map

```mermaid
mindmap
  root((U-Net paper to optical segmentation))
    Original paper
      Contracting path
      Expanding path
      Skip connections
      Valid convolutions
      Cropping
      Weighted pixel loss
      Elastic augmentation
      Overlap-tile inference
    Historical code
      Caffe
      MATLAB interface
      Ubuntu 14.04
      Pretrained network
    Modern reproduction
      PyTorch
      Padded convolutions
      Dice / BCE
      Specimen split
    Optical adaptations
      OCT layers
      MUSE tissue regions
      Fluorescence cells
      LSFM vessels
    Validation
      Dice
      Boundary accuracy
      Morphology
      Failure cases
      Domain shift
```


## 1. Why this paper is still worth reproducing

U-Net is not merely an architecture diagram. The original paper combined several ideas:

1. **contracting path** for context;
2. **expanding path** for localization;
3. **skip connections** that copy high-resolution features into the decoder;
4. **valid convolutions** (no zero padding), which shrink feature maps;
5. **cropping** of encoder features before concatenation;
6. **heavy data augmentation**, including elastic deformation;
7. **weighted pixel-wise loss** to emphasize difficult borders between touching objects;
8. **overlap-tile inference** for large images.

If you copy only the familiar ‘U-shaped network’ and omit the rest, you have implemented a U-Net **family member**, not necessarily the original paper.

### Historical reproduction question

The Freiburg release bundled trained networks, source code, MATLAB binaries/interfaces, and overlap-tile segmentation around a modified Caffe stack. It was tested on an old Linux/MATLAB environment. Therefore a serious reproduction should preserve that historical stack rather than silently rewriting it first.


## 2. Paper specification sheet

Fill this before coding:

| Field | Original-paper question |
|---|---|
| Input | microscopy image, size, channels |
| Target | pixel-wise class mask |
| Output | per-pixel class score / segmentation |
| Encoder | repeated 3×3 valid convolutions + ReLU, then 2×2 max pool |
| Decoder | up-convolution, crop+concatenate skip, repeated convolutions |
| Skip rule | crop encoder feature map to match decoder size |
| Loss | pixel-wise softmax cross-entropy with optional spatial weight map |
| Weight map | emphasize separation borders between touching objects |
| Augmentation | shifts/rotations + elastic deformation |
| Inference | overlap-tile strategy for large images |
| Split unit | must correspond to independent images/specimens in your reproduction |
| Metric | use exact benchmark metric if reproducing the challenge; use task-appropriate metrics for your own study |

## 3. Original geometry: why 572×572 becomes a smaller output

Because the original paper uses **valid** 3×3 convolutions, every convolution reduces height and width by 2 pixels.

That is why skip features must be **cropped** before concatenation.

A modern same-padding implementation usually preserves image size and avoids this crop. That is convenient, but it is a design change.


In [ ]:
def conv_valid(n, k=3):
    return n - (k - 1)

def pool2(n):
    return n // 2

def up2(n):
    return 2 * n

# Track only spatial size through the original 2D U-Net geometry.
n = 572
sizes = [("input", n)]

for level in range(4):
    n = conv_valid(conv_valid(n))
    sizes.append((f"encoder_{level+1}_after_2_convs", n))
    n = pool2(n)
    sizes.append((f"pool_{level+1}", n))

n = conv_valid(conv_valid(n))
sizes.append(("bottleneck", n))

for level in range(4):
    n = up2(n)
    sizes.append((f"decoder_{level+1}_after_up", n))
    n = conv_valid(conv_valid(n))
    sizes.append((f"decoder_{level+1}_after_2_convs", n))

print("Final spatial size:", n)
print("Expected original-paper output size for 572 input: 388")
for row in sizes[:8]:
    print(row)


The expected final spatial size is **388×388** for a 572×572 input in the canonical architecture.

### What this teaches

- Shape arithmetic is part of paper reproduction.
- A network diagram must be translated into exact tensor dimensions.
- If your code produces 572×572 output without cropping, you are using a modernized variant.

## 4. Weighted border loss: the often-forgotten idea

For touching cells, ordinary pixel loss can allow two neighboring objects to merge. The paper introduced a spatial weight map that increases the loss near narrow separation borders.

Conceptually:

```text
total pixel weight
= class-balance weight
+ border emphasis term
```

The border term becomes large where a pixel is close to the boundaries of two neighboring objects.

You do **not** have to use this exact weighting for every modern problem. But if you claim to reproduce the paper, you should understand and document it.


## 5. Historical reproduction ladder

```mermaid
flowchart TD
  A[Download Freiburg U-Net release] --> B[Record archive/version]
  B --> C[Recreate compatible Linux/Caffe/Matlab stack]
  C --> D[Run supplied segmentation script]
  D --> E[Verify sample output]
  E --> F[Locate model prototxt / weights / overlap-tile code]
  F --> G[Map each paper component to source]
  G --> H{Reference behavior reproduced?}
  H -->|No| I[Discrepancy log]
  I --> C
  H -->|Yes| J[Build modern PyTorch reimplementation]
```

Do **not** debug the historical release and modernize it at the same time.

Use two separate goals:

- **Track A — historical reproduction:** reproduce the reference release.
- **Track B — research implementation:** build a maintained PyTorch version with explicit deviations.


## 6. Modern, transparent teaching reproduction

The next example is intentionally small. It does **not** claim the paper's benchmark result.

We create synthetic optical-style objects, train a small U-Net, and measure Dice on held-out images.

This teaches image→mask supervision, encoder/decoder flow, skip concatenation, logits, BCE-with-logits, thresholding, Dice, and specimen-level separation.

For a real study, replace synthetic images with independently split specimens.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(17)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def make_sample(size=64, seed=0):
    g = torch.Generator().manual_seed(seed)
    yy, xx = torch.meshgrid(torch.arange(size), torch.arange(size), indexing="ij")
    mask = torch.zeros(size, size)
    image = torch.zeros(size, size)
    for _ in range(4):
        cy = int(torch.randint(10, size-10, (1,), generator=g))
        cx = int(torch.randint(10, size-10, (1,), generator=g))
        r = int(torch.randint(4, 9, (1,), generator=g))
        obj = ((yy-cy)**2 + (xx-cx)**2) <= r*r
        mask[obj] = 1.0
        image += obj.float() * (0.5 + 0.5*torch.rand(1, generator=g))
    image = F.avg_pool2d(image[None,None], 3, stride=1, padding=1)[0,0]
    image = image + 0.20*torch.randn(size, size, generator=g)
    image = (image - image.min()) / (image.max()-image.min()+1e-8)
    return image[None], mask[None]

X, Y = zip(*(make_sample(seed=100+i) for i in range(24)))
X, Y = torch.stack(X), torch.stack(Y)
Xtr, Ytr = X[:18], Y[:18]
Xv, Yv = X[18:], Y[18:]
print("train:", Xtr.shape, Ytr.shape)
print("valid:", Xv.shape, Yv.shape)


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1), nn.ReLU(inplace=True))
    def forward(self, x): return self.block(x)

class TinyUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.e1 = ConvBlock(1, 8)
        self.e2 = ConvBlock(8, 16)
        self.b = ConvBlock(16, 32)
        self.up2 = nn.ConvTranspose2d(32,16,2,stride=2)
        self.d2 = ConvBlock(32,16)
        self.up1 = nn.ConvTranspose2d(16,8,2,stride=2)
        self.d1 = ConvBlock(16,8)
        self.out = nn.Conv2d(8,1,1)
    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(F.max_pool2d(e1,2))
        b = self.b(F.max_pool2d(e2,2))
        d2 = self.d2(torch.cat([self.up2(b), e2], dim=1))
        d1 = self.d1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out(d1)

model = TinyUNet().to(device)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
xtr, ytr = Xtr.to(device), Ytr.to(device)
for step in range(80):
    logits = model(xtr)
    loss = F.binary_cross_entropy_with_logits(logits, ytr)
    opt.zero_grad(); loss.backward(); opt.step()
print("final train loss:", float(loss))


In [ ]:
def dice_score(pred, target, eps=1e-8):
    pred = pred.float(); target = target.float()
    inter = (pred*target).sum()
    return float((2*inter + eps) / (pred.sum()+target.sum()+eps))

with torch.no_grad():
    logits = model(Xv.to(device)).cpu()
    pred = torch.sigmoid(logits) > 0.5
scores = [dice_score(pred[i], Yv[i]) for i in range(len(Yv))]
print("validation Dice per image:", [round(s,3) for s in scores])
print("mean Dice:", round(float(np.mean(scores)),3))


## 7. What the modern example changed

| Component | Original 2015 | Teaching implementation |
|---|---|---|
| framework | modified Caffe | PyTorch |
| padding | valid | same padding |
| crop skip | yes | no |
| channels | much wider | tiny |
| classes | multi-class capable | binary |
| loss | softmax CE + optional spatial weights | BCE-with-logits |
| augmentation | strong, incl. elastic | omitted for clarity |
| inference | overlap-tile | whole 64×64 image |
| benchmark | real microscopy challenge | synthetic teaching data |

This table is the **reproducibility record**.

## 8. How to adapt U-Net to optical imaging

### OCT
Targets can include retinal layers, cornea boundaries, embryo/tissue regions, or vessel-related masks. Decide B-scan vs volume, anisotropy, log vs linear intensity, scanner domain shift, and whether region or boundary accuracy matters most.

### MUSE
Targets can include nuclei, glands, tissue/background, tumor regions, or artifact masks. Consider color channels, surface nonuniformity, staining variability, specimen-level splitting, and whether segmentation should use primary MUSE or virtual H&E.

### Fluorescence microscopy
Consider channel identity, saturation, bleaching, uneven illumination, z-projection vs 3-D, and touching-object separation.

### LSFM
Consider anisotropic z spacing, 2-D/2.5-D/3-D choice, volume memory, patch overlap, specimen-level split, and tiling artifacts.


## 9. Failure-analysis checklist

When Dice is poor, check in this order:

```text
labels correct?
→ split correct?
→ input/target aligned?
→ normalization sensible?
→ logits/activation/loss pairing correct?
→ can model overfit 1–4 samples?
→ threshold appropriate?
→ class imbalance?
→ then architecture / optimizer / augmentation
```

If a segmentation model cannot nearly memorize a few labeled images, suspect a pipeline bug before blaming generalization.

## 10. Final reproduction record

```text
paper:
original code/release:
historical environment:
modern implementation commit:
task:
independent split unit:
input:
target:
normalization:
architecture deviations:
loss:
augmentation:
optimizer:
checkpoint rule:
metric:
threshold:
failure cases:
optical-specific QC:
final decision:
```

## Final lesson

The value of U-Net is not ‘use U-Net for everything.’ The deeper lesson is to preserve localization while learning context, then validate the data/label/split pipeline before increasing model complexity.
